# 🏗️ Data Generation
> **Notebook 1 of 4** — Run top to bottom. Generates all CSVs, populates MySQL, creates analytical views, exports Tableau files.
> Outputs: `../data/generated_data/*.csv` · `../data/sql/*.sql` · `../data/tableau/*.csv`

## 0 · Imports & Connection

> Sets up the environment, connects to MySQL, and defines `save()` — the single function that writes every table to both MySQL and CSV simultaneously. Run this before anything else.

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

load_dotenv()
np.random.seed(42)

DATA_DIR    = "../data/generated_data"
TABLEAU_DIR = "../data/tableau"
FIGURES_DIR = "../figures"
SQL_DIR     = "../data/sql"

for d in [DATA_DIR, TABLEAU_DIR, FIGURES_DIR, SQL_DIR]:
    os.makedirs(d, exist_ok=True)

engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}",
    echo=False
)

# Drop all tables before regenerating (clean slate)
with engine.begin() as conn:
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 0"))
    for t in ["return_conditions", "inventory_events", "rentals",
              "rental_revenue_vs_discount", "customers", "pricing_rules",
              "products", "categories"]:
        conn.execute(text(f"DROP TABLE IF EXISTS `{t}`"))
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 1"))

def save(name, df):
    """Write DataFrame to MySQL and CSV simultaneously."""
    df.to_sql(name, engine, if_exists="replace", index=False)
    df.to_csv(f"{DATA_DIR}/{name}.csv", index=False)
    with engine.connect() as conn:
        n = conn.execute(text(f"SELECT COUNT(*) FROM `{name}`")).scalar()
    print(f"  {name}: {n:,} rows")

print(f"Connected: {os.getenv('DB_NAME')}")
print("Ready to generate.")


Connected: rental_final_project
Ready to generate.


## 1 · Categories

Each category has three fields that drive the rest of the notebook:

- `depreciation_class` — how fast the product loses value (`fast` / `standard` / `slow`). This feeds directly into the markdown discount schedule in Section 8.
- `rental_demand_tier` — how attractive the product is to renters (`high` / `medium` / `low`). Controls rental frequency and duration distributions in Section 7.
- `rental_programme` — whether this category is included in the rental programme at all. **6 categories are excluded**: Wearables (personal/hygiene), Keyboards and Peripherals (too cheap — programme overhead exceeds upside), Monitors and Networking (B2B procurement, not consumer rental behaviour). Excluding them makes the win rate more honest, not inflated.

In [2]:
categories_data = [
    # rental_programme=True  → included in rental win rate calculation
    # rental_programme=False → excluded (not a realistic rental product)
    {"category_id": 1,  "category_name": "Smartphones",         "depreciation_class": "fast",     "avg_depreciation_rate": 0.22, "rental_demand_tier": "high",   "rental_programme": True},
    {"category_id": 2,  "category_name": "Laptops",             "depreciation_class": "fast",     "avg_depreciation_rate": 0.18, "rental_demand_tier": "high",   "rental_programme": True},
    {"category_id": 3,  "category_name": "Tablets",             "depreciation_class": "fast",     "avg_depreciation_rate": 0.16, "rental_demand_tier": "medium", "rental_programme": True},
    {"category_id": 4,  "category_name": "Wearables",           "depreciation_class": "fast",     "avg_depreciation_rate": 0.20, "rental_demand_tier": "low",    "rental_programme": False},
    {"category_id": 5,  "category_name": "Drones",              "depreciation_class": "standard", "avg_depreciation_rate": 0.14, "rental_demand_tier": "high",   "rental_programme": True},
    {"category_id": 6,  "category_name": "Audio",               "depreciation_class": "standard", "avg_depreciation_rate": 0.10, "rental_demand_tier": "medium", "rental_programme": True},
    {"category_id": 7,  "category_name": "Gaming",              "depreciation_class": "standard", "avg_depreciation_rate": 0.12, "rental_demand_tier": "high",   "rental_programme": True},
    {"category_id": 8,  "category_name": "Cameras",             "depreciation_class": "standard", "avg_depreciation_rate": 0.11, "rental_demand_tier": "high",   "rental_programme": True},
    {"category_id": 9,  "category_name": "TVs",                 "depreciation_class": "slow",     "avg_depreciation_rate": 0.07, "rental_demand_tier": "low",    "rental_programme": True},
    {"category_id": 10, "category_name": "Appliances",          "depreciation_class": "slow",     "avg_depreciation_rate": 0.06, "rental_demand_tier": "low",    "rental_programme": True},
    {"category_id": 11, "category_name": "Keyboards",           "depreciation_class": "slow",     "avg_depreciation_rate": 0.05, "rental_demand_tier": "low",    "rental_programme": False},
    {"category_id": 12, "category_name": "Monitors",            "depreciation_class": "slow",     "avg_depreciation_rate": 0.08, "rental_demand_tier": "low",    "rental_programme": False},
    {"category_id": 13, "category_name": "Networking",          "depreciation_class": "standard", "avg_depreciation_rate": 0.09, "rental_demand_tier": "low",    "rental_programme": False},
    {"category_id": 14, "category_name": "Peripherals",         "depreciation_class": "slow",     "avg_depreciation_rate": 0.05, "rental_demand_tier": "low",    "rental_programme": False},
    {"category_id": 15, "category_name": "Musical Instruments", "depreciation_class": "slow",     "avg_depreciation_rate": 0.05, "rental_demand_tier": "high",   "rental_programme": True},
]
categories = pd.DataFrame(categories_data)
save("categories", categories)


  categories: 15 rows


## 2 · Pricing Rules

This table is the backbone of the A/B experiment in Notebook 3.

Two pricing models are tested:
- `flat_rate` — a fixed daily charge regardless of the product's value. Simpler for the customer; better for low-price items.
- `pct_of_retail` — a percentage of the original retail price per day. Scales with product value; better for high-end items.

Three duration models create natural groupings: weekly (`7_day`), monthly (`30_day`), and open-ended (`flexible`). Groups A and B let us compare the same pricing model under slightly different rate parameters — the A/B split.

Rate ranges are benchmarked against real consumer electronics rental platforms (Grover, Swappie): base daily rates €3–€12, insurance 2–5% of the base charge.

In [3]:
pricing_data = []
rule_id = 1
for pricing_model in ["flat_rate", "pct_of_retail"]:
    for duration_model in ["7_day", "30_day", "flexible"]:
        for experiment_group in ["A", "B"]:
            pricing_data.append({
                "rule_id":              rule_id,
                "pricing_model":        pricing_model,
                "duration_model":       duration_model,
                "experiment_group":     experiment_group,
                "base_daily_rate":      round(np.random.uniform(3, 12), 2),
                "pct_of_retail_daily":  round(np.random.uniform(0.008, 0.014), 4),
                "min_rental_days":      1 if duration_model == "flexible" else (7 if duration_model == "7_day" else 30),
                "max_rental_days":      90,
                "late_fee_per_day":     round(np.random.uniform(2, 8), 2),
                "security_deposit_pct": round(np.random.uniform(0.10, 0.25), 2),
                "insurance_fee_pct":    round(np.random.uniform(0.02, 0.05), 3),
                "created_at":           "2021-01-01",
            })
            rule_id += 1
pricing = pd.DataFrame(pricing_data)
save("pricing_rules", pricing)


  pricing_rules: 12 rows


## 3 · Seasonal Demand

Rental demand is not uniform across the year. Three multiplier tables shape how often a rental attempt succeeds in Section 7:

- `SEASONAL_HIGH` — strong November/December spike (up to 1.6×). Applies to Smartphones, Laptops, Drones, Gaming, Cameras, and Musical Instruments — the gift-season categories.
- `SEASONAL_STD` — mild Q4 uptick. Most other programme categories.
- `SEASONAL_LOW` — nearly flat, slight year-end bump. Low-demand categories like TVs and Appliances.

The multiplier is used as a probability gate in the rental loop: a month with a 0.70 multiplier is 30% less likely to produce a rental than a neutral month (1.0). This creates realistic seasonality without overriding the demand tier logic.

In [4]:
SEASONAL_STD  = {1:0.75,2:0.75,3:0.85,4:0.90,5:0.90,6:0.85,7:0.80,8:0.85,9:0.95,10:1.05,11:1.20,12:1.35}
SEASONAL_HIGH = {1:0.70,2:0.70,3:0.80,4:0.85,5:0.85,6:0.80,7:0.75,8:0.80,9:0.90,10:1.10,11:1.40,12:1.60}
SEASONAL_LOW  = {1:0.90,2:0.90,3:0.95,4:1.00,5:1.00,6:0.95,7:0.90,8:0.90,9:1.00,10:1.05,11:1.05,12:1.10}
HIGH_SEASON_CATS = {1, 2, 5, 7, 8, 15}  # Smartphones, Laptops, Drones, Gaming, Cameras, Musical Instruments

def get_seasonal_table(cat_id, demand_tier):
    if cat_id in HIGH_SEASON_CATS:
        return SEASONAL_HIGH
    elif demand_tier == "low":
        return SEASONAL_LOW
    return SEASONAL_STD

print("Seasonal tables defined.")


Seasonal tables defined.


## 4 · Products

690 products across 15 categories, sized to reflect a realistic store catalogue — high-volume categories (Keyboards, Peripherals) have more units even though they're not in the rental programme.

Key design decisions:
- `PROG_END = 2024-12-31` — the programme snapshot date. All analysis is relative to this fixed point, not today's date. This keeps the dataset stable regardless of when the notebook is run.
- Date distribution is weighted toward recent stock: 32% arrived in 2024, 36% in 2023, trailing off to 3% from 2020. A healthy store has mostly recent inventory with a long tail of slow movers.
- `rental_eligible_date = listed_date + 365 days` — products enter the programme after exactly one year unsold. This is the central assumption of the whole project.
- `condition_grade` is weighted A/B/C at 50/35/15 — most stock is in good condition; only 15% is visibly worn.

In [5]:
brands_by_cat = {
    1:["Apple","Samsung","Xiaomi","OnePlus"],
    2:["Apple","Dell","HP","Lenovo","Asus"],
    3:["Apple","Samsung","Lenovo","Huawei"],
    4:["Apple","Garmin","Fitbit","Samsung"],
    5:["DJI","Parrot","Autel"],
    6:["Sony","Bose","JBL","Sennheiser"],
    7:["Sony","Microsoft","Nintendo","Razer"],
    8:["Canon","Nikon","Sony","Fujifilm"],
    9:["Samsung","LG","Sony","Philips"],
    10:["Dyson","Bosch","Philips","Tefal"],
    11:["Logitech","Corsair","Razer","Microsoft"],
    12:["Dell","LG","Benq","AOC"],
    13:["TP-Link","Netgear","Asus","Cisco"],
    14:["Logitech","Microsoft","Anker","Belkin"],
    15:["Yamaha","Fender","Roland","Gibson","Casio"],
}

suffixes = ["Pro","Plus","Ultra","SE","X","Max","Lite",""]
PROG_END = datetime(2024, 12, 31)
n_per_cat = [
    60,  # Smartphones (high volume)
    55,  # Laptops
    40,  # Tablets
    45,  # Wearables
    20,  # Drones (low)
    50,  # Audio
    45,  # Gaming
    35,  # Cameras
    40,  # TVs
    35,  # Appliances
    65,  # Keyboards (very high)
    55,  # Monitors
    45,  # Networking
    70,  # Peripherals (highest volume)
    30,  # Musical Instruments
]

price_bands_by_cat = {
    # Smartphones
    1:  [(150, 300, 0.20), (300, 700, 0.50), (700, 1200, 0.25), (1200, 1600, 0.05)],
    # Laptops
    2:  [(500, 800, 0.25), (800, 1300, 0.45), (1300, 2000, 0.25), (2000, 3000, 0.05)],
    # Tablets
    3:  [(150, 300, 0.35), (300, 600, 0.45), (600, 1000, 0.20)],
    # Wearables
    4:  [(80, 150, 0.40), (150, 300, 0.45), (300, 600, 0.15)],
    # Drones
    5:  [(250, 500, 0.30), (500, 900, 0.45), (900, 1800, 0.25)],
    # Audio
    6:  [(40, 120, 0.35), (120, 250, 0.45), (250, 500, 0.20)],
    # Gaming
    7:  [(200, 400, 0.35), (400, 700, 0.45), (700, 1200, 0.20)],
    # Cameras (important fix)
    8:  [(300, 600, 0.35), (600, 1200, 0.40), (1200, 2200, 0.20), (2200, 4000, 0.05)],
    # TVs
    9:  [(250, 500, 0.25), (500, 900, 0.40), (900, 1600, 0.25), (1600, 3000, 0.10)],
    # Appliances
    10: [(40, 120, 0.35), (120, 300, 0.45), (300, 800, 0.20)],
    # Keyboards
    11: [(20, 50, 0.45), (50, 120, 0.40), (120, 250, 0.15)],
    # Monitors
    12: [(120, 250, 0.30), (250, 450, 0.45), (450, 900, 0.25)],
    # Networking
    13: [(25, 80, 0.35), (80, 180, 0.45), (180, 350, 0.20)],
    # Peripherals
    14: [(10, 30, 0.40), (30, 80, 0.45), (80, 150, 0.15)],
    # Musical Instruments — guitars, pianos, drums; slow depreciation, high rental demand
    15: [(80, 250, 0.30), (250, 600, 0.45), (600, 1500, 0.20), (1500, 3000, 0.05)],
}

def random_listed_date():
    # 2020 items = 4+ yrs old at PROG_END (liquidation tier)
    # 2021-mid items = 2-4 yrs old at PROG_END (heavy markdown tier)
    year = np.random.choice([2020, 2021, 2022, 2023, 2024], p=[0.03, 0.08, 0.21, 0.36, 0.32])
    if year == 2024:
        return datetime(2024, 1, 1) + timedelta(days=int(np.random.uniform(0, 181)))
    return datetime(year, 1, 1) + timedelta(days=int(np.random.uniform(0, 365)))

def sample_retail_price(category_id):
    bands = price_bands_by_cat[category_id]
    probs = [b[2] for b in bands]
    idx = np.random.choice(range(len(bands)), p=probs)
    low, high, _ = bands[idx]
    return round(np.random.uniform(low, high), 2)

products_list = []
pid = 1

for cat in categories_data:
    cid = cat["category_id"]

    for _ in range(n_per_cat[cid - 1]):
        retail = sample_retail_price(cid)
        listed = random_listed_date()
        elig = listed + timedelta(days=365)
        yrs = max(0, (PROG_END - listed).days / 365)

        dep = max(
            0.03,
            min(cat["avg_depreciation_rate"] + np.random.normal(0, 0.02), 0.30)
        )

        selected_brand = np.random.choice(brands_by_cat[cid])

        products_list.append({
            "product_id": pid,
            "category_id": cid,
            "product_name": f"{selected_brand} {cat['category_name'].rstrip('s')} {np.random.choice(suffixes)}".strip(),
            "brand": selected_brand,
            "original_retail_price": retail,
            "current_depreciated_value": round(retail * max(0.1, 1 - dep * yrs), 2),
            "condition_grade": np.random.choice(["A", "B", "C"], p=[0.5, 0.35, 0.15]),
            "listed_date": listed.date(),
            "rental_eligible_date": elig.date(),
            "retailer": np.random.choice(["FNAC Portugal", "Amazon ES"], p=[0.55, 0.45]),
            "is_active": 1,
        })

        pid += 1

products = pd.DataFrame(products_list)
save("products", products)

  products: 690 rows


## 5 · Customers

2,000 registered customers across Portugal (60%) and Spain (40%), reflecting a FNAC-style Iberian retail footprint.

Four segments with realistic weights:
- `professional` (35%) — highest repeat rental rate; tech-forward, corporate expense accounts
- `student` (25%) — price-sensitive, seasonal spikes around term start (Sep, Jan)
- `casual` (25%) — occasional renters, peak in summer and December
- `business` (15%) — fewest customers but highest rental frequency per customer

~900 customers in this dataset never converted to a rental. That's intentional — it mirrors real-world funnel drop-off and is flagged in the customer churn model (Notebook 4).

In [6]:
first_names = ["Ana","Pedro","Maria","João","Sofia","Miguel","Inês","Ricardo","Beatriz","Tiago",
               "Carlos","Luísa","Fernando","Catarina","André","Marta","Rui","Sara","Diogo","Filipa",
               "Elena","Marco","Lucia","Pablo","Rosa","Diego","Carmen","Rafael","Isabel","Nuno"]
last_names  = ["Silva","Santos","Ferreira","Pereira","Costa","Oliveira","Rodrigues","Martins",
               "Jesus","Sousa","Fernández","García","López","Martínez","González","Sánchez"]
cities_pt   = ["Lisboa","Porto","Braga","Coimbra","Setúbal","Faro","Évora","Aveiro","Funchal","Leiria"]
cities_es   = ["Madrid","Barcelona","Valencia","Sevilla","Zaragoza","Málaga","Bilbao","Alicante"]
segments    = ["student","professional","business","casual"]
seg_w       = [0.25,0.35,0.15,0.25]

customers_list = []
for cid in range(1, 2001):
    country = np.random.choice(["PT","ES"], p=[0.6,0.4])
    reg = datetime(2021,1,1) + timedelta(days=int(np.random.uniform(0,365*2)))
    customers_list.append({
        "customer_id":       cid,
        "first_name":        np.random.choice(first_names),
        "last_name":         np.random.choice(last_names),
        "city":              np.random.choice(cities_pt if country=="PT" else cities_es),
        "country":           country,
        "customer_segment":  np.random.choice(segments, p=seg_w),
        "registration_date": reg.date(),
    })
customers = pd.DataFrame(customers_list)
save("customers", customers)


  customers: 2,000 rows


## 6 · Customer Repeat Rental Pool

This solves a specific problem: if we pick customers completely at random, every product gets a different customer every time — no one ever rents twice. Real rental programmes have loyal customers.

The fix is a **weighted customer pool**. Each customer gets a number of "slots" based on their segment — business customers get 3–8 slots (they rent often), casual customers get 1–3 (they rent occasionally). The pool is shuffled and consumed round-robin, so the same customer naturally appears multiple times across the dataset.

`SEG_MONTH_BOOST` adds a soft seasonal layer on top: students are 1.6× more likely to rent in September (back-to-school), casual customers spike 1.5× in December. The boost is probabilistic — it influences, not forces.

In [7]:
SEG_RENTAL_DIST = {
    "business":     ([3,4,5,6,7,8], [0.10,0.20,0.25,0.22,0.15,0.08]),
    "professional": ([2,3,4,5],     [0.25,0.35,0.25,0.15]),
    "student":      ([1,2,3],       [0.45,0.38,0.17]),
    "casual":       ([1,2,3],       [0.55,0.33,0.12]),
}
SEG_MONTH_BOOST = {
    "student":      {1:1.40,2:1.10,9:1.60,10:1.20},
    "casual":       {6:1.20,7:1.25,8:1.20,11:1.30,12:1.50},
    "professional": {},
    "business":     {},
}
customer_pool = []
for _, row in customers.iterrows():
    vals, probs = SEG_RENTAL_DIST[row["customer_segment"]]
    n = int(np.random.choice(vals, p=probs))
    customer_pool.extend([row["customer_id"]] * n)
customer_pool = np.array(customer_pool)
np.random.shuffle(customer_pool)
pool_idx = 0
customer_segment_map = customers.set_index("customer_id")["customer_segment"].to_dict()

def next_customer(month=None):
    global pool_idx
    for _ in range(8):
        if pool_idx >= len(customer_pool):
            pool_idx = 0
            np.random.shuffle(customer_pool)
        cid = int(customer_pool[pool_idx]); pool_idx += 1
        if month is None:
            return cid
        seg   = customer_segment_map.get(cid, "casual")
        boost = SEG_MONTH_BOOST.get(seg, {}).get(month, 1.0)
        if np.random.random() < boost / 1.6:
            return cid
    if pool_idx >= len(customer_pool):
        pool_idx = 0
    cid = int(customer_pool[pool_idx]); pool_idx += 1
    return cid

print(f"Customer pool: {len(customer_pool):,} slots")


Customer pool: 5,515 slots


## 7 · Rentals, Returns & Inventory Events

The core generation loop. For each product in the rental programme, we simulate its full rental history from the day it becomes eligible until the programme end date (2024-12-31).

For each rental cycle:
1. A seasonal gate decides if a rental happens that month (low-season months are skipped more often)
2. Duration is drawn from a distribution by demand tier — high-demand items (phones, cameras) turn over fast (7–21 days); low-demand items (appliances) stay out longer (14–60 days)
3. Pricing model is assigned by price tier: items over €500 get percentage-of-retail pricing 65% of the time; items under €200 get flat-rate 72% of the time — a realistic tiered rollout
4. Revenue is calculated: base charge + late fee (if applicable) + insurance fee
5. Operational cost is 15–25% of base revenue (cleaning, inspection, software reset)
6. A 5.5% no-return rate and ~2.8% damaged-beyond-repair rate are applied independently

In [8]:
rentals_list  = []
returns_list  = []
events_list   = []
rid = 1

for _, prod in products.iterrows():
    cat_row = categories[categories["category_id"] == prod["category_id"]].iloc[0]
    if not cat_row["rental_programme"]:
        continue  # skip non-programme categories immediately

    elig = datetime.strptime(str(prod["rental_eligible_date"]), "%Y-%m-%d")
    if elig >= PROG_END:
        continue  # product not yet eligible by programme end

    days_available = (PROG_END - elig).days
    max_possible   = max(1, days_available // 30)
    n_rent = min(int(np.random.choice([4, 5, 6, 7, 8], p=[0.15, 0.25, 0.30, 0.20, 0.10])), max_possible)
    cur    = elig + timedelta(days=int(np.random.uniform(0, min(45, days_available // 2))))
    demand  = cat_row["rental_demand_tier"]
    price   = prod["original_retail_price"]
    stbl    = get_seasonal_table(int(prod["category_id"]), demand)

    for _ in range(n_rent):
        if cur >= PROG_END: break
        month = cur.month
        if np.random.random() > min(0.98, max(0.55, 0.85 * stbl[month])):
            cur += timedelta(days=int(np.random.uniform(14,30))); continue
        if demand == "high":
            dur = int(np.random.choice([7,14,21,30], p=[0.25,0.35,0.25,0.15]))
        elif demand == "medium":
            dur = int(np.random.choice([7,14,30,45], p=[0.20,0.35,0.30,0.15]))
        else:
            dur = int(np.random.choice([14,30,45,60], p=[0.20,0.35,0.30,0.15]))
        if month in (11,12) and demand == "high":
            dur = int(np.random.choice([7,14,21], p=[0.40,0.35,0.25]))
        end_dt = cur + timedelta(days=dur)
        # Pricing model assignment: expensive items → % of retail (scales with value)
        # Cheap items → flat rate (simpler, more predictable for the customer)
        if price > 500 and np.random.random() < 0.65:
            pm = "pct_of_retail"
        elif price < 200 and np.random.random() < 0.72:
            pm = "flat_rate"
        elif np.random.random() < 0.58:
            pm = "flat_rate"
        else:
            pm = "pct_of_retail"
        rule = pricing[pricing["pricing_model"]==pm].sample(1).iloc[0]
        base_rev = round(rule["base_daily_rate"]*dur, 2) if pm=="flat_rate"                    else round(rule["pct_of_retail_daily"]*price*dur, 2)
        if month in (11,12): base_rev = round(base_rev*np.random.uniform(1.05,1.15),2)
        is_late  = np.random.random() < np.random.uniform(0.10,0.15)
        late_d   = int(np.random.uniform(1,8)) if is_late else 0
        late_fee = round(rule["late_fee_per_day"]*late_d, 2) if is_late else 0.0
        ins_fee  = round(base_rev*rule["insurance_fee_pct"], 2)
        total    = round(base_rev+late_fee+ins_fee, 2)
        op_cost  = round(base_rev*round(np.random.uniform(0.15,0.25),3), 2)
        net_rev  = round(total-op_cost, 2)
        no_ret   = np.random.random() < 0.055
        dbr      = no_ret and (np.random.random() < 0.50)
        exp_ret  = end_dt + timedelta(days=late_d)
        act_ret  = exp_ret if not no_ret else None
        rentals_list.append({
            "rental_id":rid,"product_id":int(prod["product_id"]),"customer_id":next_customer(month=month),
            "pricing_rule_id":int(rule["rule_id"]),"rental_start_date":cur.date(),"rental_end_date":end_dt.date(),
            "expected_return_date":exp_ret.date(),"actual_return_date":act_ret.date() if act_ret else None,
            "rental_duration_days":dur,"base_rental_revenue":base_rev,"late_fee":late_fee,"insurance_fee":ins_fee,
            "total_rental_revenue":total,"operational_cost":op_cost,"net_rental_revenue":net_rev,
            "is_no_return":int(no_ret),"is_damaged_beyond_repair":int(dbr),"is_late":int(is_late),
        })
        if not no_ret:
            returns_list.append({"rental_id":rid,"product_id":int(prod["product_id"]),
                "condition_on_return":np.random.choice(["excellent","good","fair","damaged"],p=[0.30,0.45,0.18,0.07]),
                "damage_fee":round(np.random.uniform(0,50),2) if np.random.random()<0.15 else 0.0,"return_note":""})
        events_list.append({"event_id":rid,"product_id":int(prod["product_id"]),
            "event_type":"rental_start","event_date":cur.date(),"notes":f"rental_id={rid}"})
        rid += 1
        cur = (act_ret or exp_ret) + timedelta(days=int(np.random.uniform(5,15)))

rentals = pd.DataFrame(rentals_list)
returns = pd.DataFrame(returns_list)
events  = pd.DataFrame(events_list)
save("rentals", rentals)
save("return_conditions", returns)
save("inventory_events", events)

  rentals: 1,093 rows
  return_conditions: 1,047 rows
  inventory_events: 1,093 rows


## 8 · Rental vs Discount Comparison

This is the table that answers the central question: for each product, did the rental programme generate more revenue than a clearance markdown would have?

The markdown discount is calculated using a depreciation schedule sourced from real electronics resale data (SellCell, EverTrade IT asset curves):
- `fast` class (phones, laptops): 45% off at 12 months → 75% off at 24+ months
- `standard` class (drones, cameras): 35% off at 12 months → 72% off at 24+ months
- `slow` class (TVs, appliances): 25% off at 12 months → 65% off at 24+ months

One important rule on the rental side: if the last rental ended in **damaged beyond repair**, that final cycle's net revenue is zeroed out (the product was lost). All prior rentals still count. This avoids double-penalising a product for a single bad outcome.

Only `rental_programme = True` categories appear in this table — win rate is scoped to products that were actually eligible.

In [9]:
def get_discount(months_unsold, dep_class):
    # Discount % applied to retail price at each age threshold.
    # At 24+ months electronics hit liquidation: retailer keeps ≤30% of retail.
    # Sources: SellCell depreciation data; EverTrade IT asset curves.
    tiers = {
        #            months  discount_pct (% OFF retail)
        "fast":     [(12,.45),(18,.55),(24,.70),(999,.75)],
        "standard": [(12,.35),(18,.45),(24,.65),(999,.72)],
        "slow":     [(12,.25),(18,.35),(24,.55),(999,.65)],
    }
    for thr, pct in tiers[dep_class]:
        if months_unsold <= thr: return pct
    return 0.72

comparison_list = []
for _, prod in products.iterrows():
    pid    = int(prod["product_id"])
    listed = datetime.strptime(str(prod["listed_date"]), "%Y-%m-%d")
    elig   = datetime.strptime(str(prod["rental_eligible_date"]), "%Y-%m-%d")
    months_unsold = (PROG_END - listed).days / 30.44
    cat_row   = categories[categories["category_id"]==prod["category_id"]].iloc[0]
    if not cat_row["rental_programme"]:
        continue  # exclude non-programme categories from win rate calculation
    disc_pct  = get_discount(months_unsold, cat_row["depreciation_class"])
    disc_price = round(prod["original_retail_price"]*(1-disc_pct), 2)
    prod_r = rentals[rentals["product_id"]==pid]
    n_rents = len(prod_r)
    if n_rents > 0:
        net_rev = round(prod_r.iloc[:-1]["net_rental_revenue"].sum(), 2)                   if prod_r.iloc[-1]["is_damaged_beyond_repair"]                   else round(prod_r["net_rental_revenue"].sum(), 2)
    else:
        net_rev = 0.0
    gross_rev = prod_r["total_rental_revenue"].sum() if n_rents > 0 else 0.0
    op_cost   = prod_r["operational_cost"].sum()      if n_rents > 0 else 0.0
    avg_dur   = prod_r["rental_duration_days"].mean() if n_rents > 0 else 0
    months_on = round(n_rents*avg_dur/30.44, 2)       if n_rents > 0 else 0
    ratio = round(net_rev/disc_price, 4) if disc_price > 0 else 0.0
    comparison_list.append({
        "product_id":                  pid,
        "original_retail_price":       prod["original_retail_price"],
        "months_at_enrollment":        round((elig-listed).days/30.44, 1),
        "months_unsold_at_comparison": round(months_unsold, 1),
        "discount_pct":                disc_pct,
        "hypothetical_discount_price": disc_price,
        "total_gross_rental_revenue":  round(gross_rev, 2),
        "total_operational_cost":      round(op_cost, 2),
        "total_net_rental_revenue":    round(net_rev, 2),
        "n_rentals":                   n_rents,
        "months_on_rental":            months_on,
        "rental_vs_discount_ratio":    ratio,
        "is_rental_more_profitable":   int(ratio > 1.0),
    })
comparison = pd.DataFrame(comparison_list)
save("rental_revenue_vs_discount", comparison)


  rental_revenue_vs_discount: 410 rows


## 9 · Summary

Quick sanity check before writing to MySQL. Key numbers to watch:
- **Win rate** should land 60–65%
- **Median ratio** is the honest number — mean is skewed by 2020/2021 products with 4+ years of rental history
- **No-return rate** should be 5–7%, late returns 10–15%

In [10]:
win_rate    = comparison["is_rental_more_profitable"].mean() * 100
mean_ratio  = comparison["rental_vs_discount_ratio"].mean()
median_ratio = comparison["rental_vs_discount_ratio"].median()
avg_rents   = rentals.groupby("product_id").size().mean()
no_ret      = rentals["is_no_return"].mean() * 100
dbr_rate    = rentals["is_damaged_beyond_repair"].mean() * 100
late_rate   = rentals["is_late"].mean() * 100

print("=" * 50)
print("DATA GENERATION SUMMARY")
print("=" * 50)
print(f"Products:              {len(products)}")
print(f"Customers:             {len(customers):,}")
print(f"Rentals:               {len(rentals):,}")
print(f"Returns:               {len(returns):,}")
print(f"Date range:            {rentals['rental_start_date'].min()} to {rentals['rental_start_date'].max()}")
print(f"Avg rentals/product:   {avg_rents:.1f}")
print(f"No-return rate:        {no_ret:.1f}%")
print(f"Damaged beyond repair: {dbr_rate:.1f}%")
print(f"Late return rate:      {late_rate:.1f}%")
print(f"Rental win rate:       {win_rate:.1f}%")
print(f"Median ratio (honest): {median_ratio:.2f}x")
print(f"Mean ratio (skewed):   {mean_ratio:.2f}x  ← inflated by 2020/2021 products")
print("=" * 50)

DATA GENERATION SUMMARY
Products:              690
Customers:             2,000
Rentals:               1,093
Returns:               1,047
Date range:            2021-01-21 to 2024-12-28
Avg rentals/product:   3.8
No-return rate:        4.2%
Damaged beyond repair: 1.6%
Late return rate:      10.4%
Rental win rate:       55.1%
Median ratio (honest): 1.27x
Mean ratio (skewed):   2.61x  ← inflated by 2020/2021 products


## 10 · MySQL Analytical Views

In [11]:
views = {
    "v_rental_eligible": (
        "SELECT p.product_id, p.product_name, p.brand, p.original_retail_price, "
        "p.current_depreciated_value, p.listed_date, p.rental_eligible_date, "
        "p.condition_grade, c.category_name, c.depreciation_class, c.rental_demand_tier "
        "FROM products p "
        "JOIN categories c ON p.category_id = c.category_id "
        "WHERE p.rental_eligible_date <= '2024-12-31'"
    ),
    "v_rental_history": (
        "SELECT r.rental_id, r.product_id, r.customer_id, r.rental_start_date, "
        "r.rental_end_date, r.actual_return_date, r.rental_duration_days, "
        "r.total_rental_revenue, r.net_rental_revenue, r.operational_cost, "
        "r.is_late, r.is_no_return, r.is_damaged_beyond_repair, "
        "p.product_name, p.brand, c.category_name, cu.customer_segment, cu.city, cu.country "
        "FROM rentals r "
        "JOIN products p  ON r.product_id  = p.product_id "
        "JOIN categories c ON p.category_id = c.category_id "
        "JOIN customers cu ON r.customer_id  = cu.customer_id"
    ),
    "v_inventory_aging": (
        "SELECT p.product_id, p.product_name, p.brand, p.listed_date, p.rental_eligible_date, "
        "DATEDIFF('2024-12-31', p.listed_date) AS days_on_shelf, "
        "p.original_retail_price, p.current_depreciated_value, "
        "c.category_name, c.depreciation_class "
        "FROM products p "
        "JOIN categories c ON p.category_id = c.category_id"
    ),
    "v_revenue_comparison": (
        "SELECT rv.product_id, p.product_name, p.brand, "
        "c.category_name, c.depreciation_class, "
        "rv.months_unsold_at_comparison, rv.discount_pct, rv.hypothetical_discount_price, "
        "rv.total_gross_rental_revenue, rv.total_operational_cost, rv.total_net_rental_revenue, "
        "rv.rental_vs_discount_ratio, rv.is_rental_more_profitable, rv.n_rentals, rv.months_on_rental "
        "FROM rental_revenue_vs_discount rv "
        "JOIN products p  ON rv.product_id  = p.product_id "
        "JOIN categories c ON p.category_id  = c.category_id"
    ),
}

with engine.begin() as conn:
    for name, select_sql in views.items():
        conn.execute(text(f"DROP VIEW IF EXISTS `{name}`"))
        conn.execute(text(f"CREATE VIEW `{name}` AS {select_sql}"))

with engine.connect() as conn:
    for name in views:
        n = conn.execute(text(f"SELECT COUNT(*) FROM `{name}`")).scalar()
        print(f"  {name}: {n:,} rows")

# Export view DDL as .sql reference files
for name, select_sql in views.items():
    path = SQL_DIR + "/" + name + ".sql"
    with open(path, "w") as f:
        f.write("DROP VIEW IF EXISTS `" + name + "`;\n")
        f.write("CREATE VIEW `" + name + "` AS " + select_sql + ";\n")
print("Views created. DDL saved to " + SQL_DIR + "/")

  v_rental_eligible: 476 rows
  v_rental_history: 1,093 rows
  v_inventory_aging: 690 rows
  v_revenue_comparison: 410 rows
Views created. DDL saved to ../data/sql/


## 11 · Tableau Export

In [12]:
tableau_data = (
    rentals
    .merge(products[["product_id","category_id","product_name","brand",
                      "original_retail_price","current_depreciated_value",
                      "listed_date","rental_eligible_date","condition_grade"]], on="product_id")
    .merge(categories[["category_id","category_name","avg_depreciation_rate","rental_demand_tier"]], on="category_id")
    .merge(customers[["customer_id","first_name","last_name","city","country","customer_segment"]], on="customer_id")
    .merge(pricing[["rule_id","pricing_model","duration_model","experiment_group"]],
           left_on="pricing_rule_id", right_on="rule_id", how="left")
    .merge(returns[["rental_id","condition_on_return","damage_fee"]], on="rental_id", how="left")
    .merge(comparison[["product_id","hypothetical_discount_price","total_net_rental_revenue",
                        "total_operational_cost","rental_vs_discount_ratio","is_rental_more_profitable"]],
           on="product_id", how="left")
)
tableau_data["customer_name"] = tableau_data["first_name"] + " " + tableau_data["last_name"]
tableau_data.drop(columns=["first_name","last_name","rule_id"], inplace=True, errors="ignore")
tableau_data.to_csv(f"{TABLEAU_DIR}/rental_analysis_full.csv", index=False)
print(f"rental_analysis_full.csv: {len(tableau_data):,} rows, {len(tableau_data.columns)} columns")

profitability = comparison.merge(
    products[["product_id","product_name","brand","category_id"]], on="product_id"
).merge(categories[["category_id","category_name"]], on="category_id")
profitability.to_csv(f"{TABLEAU_DIR}/product_profitability.csv", index=False)
print(f"product_profitability.csv: {len(profitability)} rows")
print(f"Tableau files saved to {TABLEAU_DIR}/")


rental_analysis_full.csv: 1,093 rows, 43 columns
product_profitability.csv: 410 rows
Tableau files saved to ../data/tableau/


---
## Done

Run top to bottom. Every section prints row counts as it goes.
Proceed to `02_eda.ipynb`.